In [1]:
# === Setup: carregar scripts oficiais do GitHub ===
!rm -rf /content/pub-ensino-ht
!git clone --depth 1 https://github.com/TarTaRogue/pub-ensino-ht.git /content/pub-ensino-ht

import sys
sys.path.insert(0, "/content/pub-ensino-ht")
sys.path.insert(0, ".")   # fallback: arquivos enviados junto ao notebook

try:
    import pubensino.ht.natconv_iso as _t
    print("OK: scripts carregados do GitHub")
except ModuleNotFoundError:
    print("ATENCAO: modulo natconv_iso nao encontrado no repositorio clonado.\n"
          "Adicione os arquivos novos (natconv_iso.py, viz_natconv.py, widgets_natconv.py)\n"
          "em pubensino/ht/ ao repositorio e faca push, ou envie-os junto a este notebook.")

# scipy e ipywidgets ja estao disponiveis no Colab.
from IPython.display import HTML
HTML("""
<style>
div.input {display:none;}
</style>
""")


Cloning into '/content/pub-ensino-ht'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 42 (delta 1), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 572.30 KiB | 6.22 MiB/s, done.
Resolving deltas: 100% (1/1), done.
OK: scripts carregados do GitHub


# Notebook 4 — Convecção Natural em Superfície Isotérmica

**Objetivo.** Demonstrar os conceitos da convecção natural (livre) sobre superfícies isotérmicas e a aplicação tanto da **solução das equações de transporte** (solução de similaridade da camada-limite) quanto das **correlações empíricas** de número de Nusselt. A ferramenta permite explorar, de forma interativa, como as condições físicas (fluido, diferença de temperatura, geometria e dimensão) controlam os grupos adimensionais, a estrutura da camada-limite e o coeficiente de transferência de calor.

## Hipóteses físicas e escopo

- Regime permanente.
- Escoamento induzido **apenas** por empuxo (sem velocidade forçada imposta).
- Aproximação de Boussinesq: propriedades constantes, exceto a massa específica no termo de empuxo, linearizada por $\rho \approx \rho_\infty\,[1-\beta(T-T_\infty)]$.
- Propriedades termofísicas avaliadas na **temperatura de filme** $T_f=(T_s+T_\infty)/2$.
- Superfície **isotérmica** ($T_s$ uniforme); fluido ambiente em repouso a $T_\infty$.
- Para a solução de similaridade: camada-limite laminar, bidimensional, sobre placa vertical.
- Radiação desprezível.

Essas hipóteses permitem (i) reduzir as equações de transporte a um sistema de equações diferenciais ordinárias acopladas via variável de similaridade e (ii) empregar correlações clássicas de $\mathrm{Nu}(\mathrm{Ra},\mathrm{Pr})$.

In [2]:
import importlib
import pubensino.ht.natconv_iso as natconv_iso
import pubensino.ht.viz_natconv as viz_natconv
import pubensino.ht.widgets_natconv as widgets_natconv

## Fundamentos teóricos

Na convecção natural, o movimento do fluido não é imposto externamente: ele surge de **forças de empuxo** geradas por gradientes de massa específica que, por sua vez, decorrem de gradientes de temperatura em presença de um campo gravitacional. Junto a uma superfície aquecida vertical, o fluido próximo se aquece, torna-se menos denso e sobe, formando uma **camada-limite** de velocidade e térmica que cresce ao longo da superfície.

O parâmetro que mede a importância relativa do empuxo frente aos efeitos viscosos é o número de **Grashof**,
$$
\mathrm{Gr}_L = \frac{g\,\beta\,(T_s-T_\infty)\,L^3}{\nu^2},
$$
e o parâmetro que governa diretamente o início e a intensidade da convecção é o número de **Rayleigh**,
$$
\mathrm{Ra}_L = \mathrm{Gr}_L\,\mathrm{Pr} = \frac{g\,\beta\,(T_s-T_\infty)\,L^3}{\nu\,\alpha},
$$
com $\mathrm{Pr}=\nu/\alpha$. Para gás ideal, $\beta = 1/T_f$.

O resultado de engenharia é o número de **Nusselt** médio, que adimensionaliza o coeficiente convectivo:
$$
\overline{\mathrm{Nu}}_L = \frac{\bar h\,L}{k}\quad\Rightarrow\quad \bar h = \frac{\overline{\mathrm{Nu}}_L\,k}{L},\qquad q'' = \bar h\,(T_s-T_\infty).
$$

Na placa vertical, o escoamento permanece laminar até cerca de $\mathrm{Ra}_L\approx 10^{9}$, transicionando para turbulento acima desse valor.

## Modelo matemático

**Equações de camada-limite (placa vertical, Boussinesq).** Conservação de massa, quantidade de movimento na direção do escoamento $x$ e energia:
$$
\frac{\partial u}{\partial x}+\frac{\partial v}{\partial y}=0,
$$
$$
u\frac{\partial u}{\partial x}+v\frac{\partial u}{\partial y}= g\,\beta\,(T-T_\infty)+\nu\frac{\partial^2 u}{\partial y^2},
$$
$$
u\frac{\partial T}{\partial x}+v\frac{\partial T}{\partial y}= \alpha\frac{\partial^2 T}{\partial y^2}.
$$

**Transformação de similaridade (Ostrach).** Definindo
$$
\eta=\left(\frac{\mathrm{Gr}_x}{4}\right)^{1/4}\frac{y}{x},\qquad
\theta(\eta)=\frac{T-T_\infty}{T_s-T_\infty},
$$
e uma função corrente que conduz a $f(\eta)$, o sistema parcial reduz-se a duas EDOs acopladas:
$$
f''' + 3 f f'' - 2\,(f')^2 + \theta = 0,
$$
$$
\theta'' + 3\,\mathrm{Pr}\,f\,\theta' = 0,
$$
com condições de contorno
$$
f(0)=f'(0)=0,\quad \theta(0)=1,\qquad f'(\infty)=0,\quad \theta(\infty)=0.
$$
A componente de velocidade vale $u=\dfrac{2\nu}{x}\,\mathrm{Gr}_x^{1/2}\,f'(\eta)$, de modo que $f'(\eta)$ representa o **perfil de velocidade adimensional** — característico por subir junto à parede e retornar a zero longe dela. O número de Nusselt local resulta do gradiente de temperatura na parede:
$$
\mathrm{Nu}_x=\left(\frac{\mathrm{Gr}_x}{4}\right)^{1/4}\big[-\theta'(0)\big],\qquad
\overline{\mathrm{Nu}}_L=\tfrac{4}{3}\,\mathrm{Nu}_{x=L}\ \text{(laminar)}.
$$

**Correlações empíricas.** Para uso geral em projeto, empregam-se correlações de $\mathrm{Nu}(\mathrm{Ra},\mathrm{Pr})$. Para a placa vertical isotérmica (Churchill & Chu, válida em toda a faixa de $\mathrm{Ra}$):
$$
\overline{\mathrm{Nu}}_L=\left\{0{,}825+\frac{0{,}387\,\mathrm{Ra}_L^{1/6}}{\left[1+(0{,}492/\mathrm{Pr})^{9/16}\right]^{8/27}}\right\}^2 .
$$
Correlações análogas existem para cilindro horizontal, esfera e placa horizontal (exploradas na segunda ferramenta).

## Fixação de conceitos

* Na solução de similaridade de Blasius (convecção forçada), a EDO resultante não é acoplada à equação da energia. Em contrapartida, na convecção natural, esse acoplamento ocorre. Por quê?
* Com base na transformação sugerida, deduza as duas EDOs acopladas (sugestão: inicie com a derivada parcial de $f(\eta)$, aplicando a regra da cadeia).
* Como as condições de contorno são estabelecidas a partir da física do problema?

## Placa vertical isotérmica

Ajuste o fluido, as temperaturas e a altura $L$ da placa. O painel reúne a configuração física, os perfis de similaridade (velocidade e temperatura), o crescimento da camada-limite e a curva $\mathrm{Nu}\times\mathrm{Ra}$ com o ponto de operação. No regime laminar, o $\mathrm{Nu}$ obtido pela **solução de similaridade** é comparado ao da **correlação**.

In [3]:
import importlib
import pubensino.ht.widgets_natconv as widgets_natconv
importlib.reload(widgets_natconv)

widgets_natconv.vertical_plate()

Output()

## Discussão física

- O **perfil de velocidade** $f'(\eta)$ parte de zero na parede (não escorregamento), cresce até um máximo dentro da camada-limite e **retorna a zero** na borda — diferentemente da convecção forçada, em que a velocidade tende à corrente livre. Esse formato é a assinatura da convecção natural.
- Aumentar $T_s-T_\infty$ ou a altura $L$ eleva fortemente $\mathrm{Ra}_L$ (dependência $\propto L^3$), espessando a camada-limite e elevando $\bar h$ — porém de modo sublinear, pois $\overline{\mathrm{Nu}}_L\sim\mathrm{Ra}^{1/4}$ (laminar) e $\sim\mathrm{Ra}^{1/3}$ (turbulento).
- Para $\mathrm{Pr}$ maior (líquidos), a camada térmica fica mais fina e $-\theta'(0)$ aumenta, elevando o Nusselt.
- Próximo a $\mathrm{Ra}_L\approx10^{9}$ ocorre a **transição** laminar→turbulento, marcada no gráfico $\mathrm{Nu}\times\mathrm{Ra}$.
- No regime laminar, $\mathrm{Nu}$ da similaridade e da correlação coincidem dentro de poucos por cento — validação cruzada dos dois métodos.

## Comparação de geometrias isotérmicas

Para um mesmo fluido, $\Delta T$ e comprimento característico $L_c$, compare as correlações de placa vertical, cilindro horizontal, esfera e placa horizontal (faces para cima e para baixo). A tabela mostra $\mathrm{Nu}$, $\bar h$ e $q''$ de cada geometria; o gráfico sobrepõe as curvas $\mathrm{Nu}\times\mathrm{Ra}$ com os pontos de operação.

In [4]:
import importlib
import pubensino.ht.widgets_natconv as widgets_natconv
importlib.reload(widgets_natconv)

widgets_natconv.geometry_comparison()

**Correlações de Nu aplicadas (convecção natural, superfícies isotérmicas):**



**Placa vertical (L = altura)**

$$Nu = \left\{0{,}825 + \dfrac{0{,}387\,Ra^{1/6}}{\left[1+(0{,}492/Pr)^{9/16}\right]^{8/27}}\right\}^{2}$$

Faixa de validade: 0.1 ≤ Ra ≤ 1e+13 &nbsp;•&nbsp; todo Pr

**Cilindro horizontal (L = D)**

$$Nu = \left\{0{,}60 + \dfrac{0{,}387\,Ra^{1/6}}{\left[1+(0{,}559/Pr)^{9/16}\right]^{8/27}}\right\}^{2}$$

Faixa de validade: 1e-05 ≤ Ra ≤ 1e+12 &nbsp;•&nbsp; todo Pr

**Esfera (L = D)**

$$Nu = 2 + \dfrac{0{,}589\,Ra^{1/4}}{\left[1+(0{,}469/Pr)^{9/16}\right]^{4/9}}$$

Faixa de validade: 1 ≤ Ra ≤ 1e+11 &nbsp;•&nbsp; Pr ≥ 0.7

**Placa horizontal, face quente p/ cima (L = A/P)**

$$Nu = 0{,}54\,Ra^{1/4}\ (10^{4}\!\le\!Ra\!\le\!10^{7});\ \ Nu = 0{,}15\,Ra^{1/3}\ (10^{7}\!\le\!Ra\!\le\!10^{11})$$

Faixa de validade: 10000 ≤ Ra ≤ 1e+11 &nbsp;•&nbsp; Pr ≥ 0.7

**Placa horizontal, face quente p/ baixo (L = A/P)**

$$Nu = 0{,}27\,Ra^{1/4}\ (10^{5}\!\le\!Ra\!\le\!10^{10})$$

Faixa de validade: 100000 ≤ Ra ≤ 1e+10 &nbsp;•&nbsp; Pr ≥ 0.7

_Lc é o comprimento característico de cada geometria; as propriedades entram via Ra e Pr. Fonte: Bergman/Incropera — confira constantes e faixas com o livro do curso._

Output()

## Limites do modelo

- Propriedades dependentes da temperatura (variações fortes de $\Delta T$) e efeitos de compressibilidade.
- Validade da aproximação de Boussinesq apenas para $\Delta T$ moderado frente a $T_f$.
- A solução de similaridade pressupõe regime **laminar e bidimensional**; não descreve a região turbulenta nem efeitos de borda.
- Superfície idealizada como perfeitamente isotérmica (fluxo de calor não uniforme exigiria outra formulação).
- Contribuição radiativa, interação com paredes próximas e convecção mista (empuxo + escoamento forçado) não são consideradas.
- As correlações têm faixas de validade em $\mathrm{Ra}$ e $\mathrm{Pr}$ que devem ser respeitadas.